[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyneuro/Chapter_Colabs/blob/main/Colab_E.ipynb)


# Set E — Synapses & Two‑Neuron Communication (LEGO–Colab)
**Author: Neural Engineering Laboratory, University of Missouri - Gregory Glickert, Khuram Choudhry, Ziao Chen, Satish S. Nair

**Run order:** E0 ▶ E1 ▶ … ▶ E9. If things desync: *Runtime ▶ Restart & run all*.  

---
## Table of Contents
- [E0 — Starter](#e0)
- [E1 — Two neurons + AMPA‑like synapse](#e1)
- [E2 — Inhibitory (GABA_A‑like)](#e2)
- [E3 — EPSP vs IPSP (peaks & latencies)](#e3)
- [E4 — Exercise: excitatory vs inhibitory? (E_rev sweep)](#e4)
- [E5 — Temporal summation (two spikes, variable ISI)](#e5)
- [E6 — Spatial summation (proximal vs distal)](#e6)
- [E7 — Feedforward inhibition (E then I)](#e7)
- [E8 — Analysis helpers (latency, peak, area)](#e8)
- [E9 — Playground](#e9)
- [Reflection](#reflection)
---


### **E0 — Starter**

**Purpose**
Initialize the NEURON environment, load plotting helpers, and define a recorder factory (`mk_rec`) to confirm the environment is fully operational for electrophysiology simulations.

---

### **Post-Run Checklist**
After executing the initialization code, please record the following:
*   **NEURON Version:** Note the specific version string printed during import.
*   **Global Defaults:** List the default values used for the simulation (e.g., $R_a$, $c_m$).
*   **Functionality Check:** Confirm that the helper functions `mk_rec()` and `plot_vm()` are loaded and working correctly.

---

### **Exercises**

#### **1. Experimental Definitions**
In one concise sentence each, define the following:
*   **Current Clamp:** You command a specific current ($I$) and observe the resulting changes in membrane potential ($V_m$).
*   **Conductance Injection:** You command a time-varying conductance ($g(t)$) and observe how the membrane potential ($V_m$) responds.

#### **2. Reproducibility Header**
Create a new Markdown cell at the very top of your notebook labeled **"Reproducibility Header"** and include the following:
*   **Metadata:** Author name and current date.
*   **Conventions:** State the unit conventions being used (e.g., ms, mV, nA, $\text{M}\Omega$).
*   **Documentation Policy:** Briefly state your policy for labeling figures (e.g., "All figures must include axes labels with units, a legend where applicable, and a descriptive title").

#### **3. Data Presentation**
**Question:** Explain why including clear **axis units** and a **legend** is strictly necessary for the grading and peer review of computational notebooks.
*   [Your Answer Here]


In [ ]:

!pip -q install neuron==8.2.4
try:
    from neuron import h, gui
except Exception:
    from neuron import h
from neuron.units import ms, mV
import numpy as np
import matplotlib.pyplot as plt
h.load_file('stdrun.hoc')

s0=h.Section(name='bootstrap')

def plot_vm(t_vec,traces,labels,title='',figsize=(7,3.3)):
    plt.figure(figsize=figsize)
    for tr,lb in zip(traces,labels):
        plt.plot(t_vec,tr,label=lb)
    plt.xlabel('Time (ms)'); plt.ylabel('Voltage (mV)'); plt.title(title)
    plt.grid(True,alpha=0.3);
    if len(labels)>1: plt.legend()
    plt.tight_layout(); plt.show()

t = h.Vector(); t.record(h._ref_t)

def mk_rec(sec):
    v=h.Vector(); v.record(sec(0.5)._ref_v); return v

print('E0 ready. Proceed to E1.')


### **E1 — Two neurons + AMPA-like excitatory synapse**

**Idea**
A presynaptic spike triggers an `ExpSyn` conductance on the postsynaptic soma. This synapse mimics **AMPA-like** receptors with a reversal potential near **0 mV**.

---

### **What to change**
*   **Synaptic Strength:** `nc.weight[0]` (the weight of the NetCon connection).
*   **Synaptic Kinetics:** `syn.tau` (the time constant of the conductance decay).
*   **Axonal Latency:** `nc.delay` (the time between the presynaptic spike and the postsynaptic response).

---

### **Sanity checks**
*   **Dual Recording:** Plot the presynaptic and postsynaptic membrane potentials ($V_m$) on the same axes.
*   **Feature Extraction:** Report the calculated **EPSP peak** amplitude and the **synaptic latency** (time to peak).

---

### **Predict → verify**
*   **Weight Scaling:** Increasing the weight should scale the EPSP peak approximately linearly for small, sub-threshold amplitudes.
*   **Kinetic Slowing:** Increasing the time constant ($\tau$) should make the EPSP broader and shift the peak to a later time point.

---

### **Exercises**

**Kinetic Sweep**
Perform a sweep of the synaptic time constant $\tau \in \{1, 2, 5\text{ ms}\}$ while holding the weight constant. 
*   **Report:** Create a small table for each $\tau$ value listing the resulting **EPSP peak** and **latency**.

**Linearity Boundary**
Double the initial synaptic weight. 
*   **Identify:** At what point does the relationship between peak amplitude and weight begin to deviate from linearity (sublinear summation)? 
*   **Explain:** Why does this "saturation" occur as the membrane potential approaches the synaptic reversal potential?


In [ ]:

somaA=h.Section(name='somaA'); somaA.L=somaA.diam=20; somaA.Ra=100; somaA.cm=1; somaA.insert('hh')
somaB=h.Section(name='somaB'); somaB.L=somaB.diam=20; somaB.Ra=100; somaB.cm=1; somaB.insert('hh')

iclampA=h.IClamp(somaA(0.5)); iclampA.delay=5; iclampA.dur=2; iclampA.amp=0.4
syn=h.ExpSyn(somaB(0.5)); syn.tau=2.0; syn.e=0.0
nc=h.NetCon(somaA(0.5)._ref_v,syn,sec=somaA)
nc.threshold=-20; nc.delay=1.0; nc.weight[0]=0.005
vA=mk_rec(somaA); vB=mk_rec(somaB)
h.finitialize(-65*mV); h.continuerun(30*ms)
plot_vm(t,[vA,vB],["Neuron A","Neuron B"],title='E1: A→B excitatory (AMPA-like)')


### **E2 — Switch to inhibitory synapse and GABAA-like inhibition**

**Idea**  
Transition the synapse to a **GABAA-like inhibitory** profile. This involves shifting the reversal potential to a hyperpolarized level and extending the decay time constant to reflect slower inhibitory kinetics.

---

### **What to change**
*   **Reversal Potential:** Set `syn.e` to approximately **−75 mV**.
*   **Synaptic Kinetics:** Increase `syn.tau` (e.g., to a range of **6–10 ms**).
*   **Synaptic Strength:** Adjust `nc.weight[0]` to produce a clearly visible IPSP.

---

### **Sanity checks**
*   **IPSP Polarity:** Confirm that the IPSP peak is negative (hyperpolarizing) relative to a resting baseline of approximately **−65 mV**.
*   **Trace Duration:** Ensure the simulation runs long enough to capture the full decay of the slower inhibitory conductance.

---

### **Predict → verify**
*   **Driving Force:** As the membrane potential ($V_m$) is made more negative, the magnitude of the outward inhibitory current should decrease.
*   **Shunting Inhibition:** When $V_m$ is held near the reversal potential ($E_{rev}$), the synapse produces little to no voltage change but still suppresses excitability by "shunting" the membrane (increasing total conductance).

---

### **Exercises**

**Driving Force Analysis**
Hold the cell at **−70 mV** and then at **−50 mV** before triggering the synapse. 
*   **Explain:** Contrast the sign and magnitude of the resulting synaptic responses based on the electrochemical driving force.

**Functional Suppression (Shunting)**
Demonstrate a scenario where the resting potential is slightly more negative than the reversal potential (e.g., rest at **−80 mV**). 
*   **Observe:** Show that the resulting trace "depolarizes" toward the reversal potential, yet the synapse remains functionally inhibitory by reducing the input resistance and making it harder for other inputs to reach threshold.


In [ ]:

syn.e=-75.0; syn.tau=8.0; nc.weight[0]=0.01
h.finitialize(-65*mV); h.continuerun(40*ms)
plot_vm(t,[vA,vB],["Neuron A","Neuron B"],title='E2: A→B inhibitory (GABA_A-like)')


### **E3 — EPSP vs IPSP contrast peaks and latencies**

**Idea**
Run the same protocol for excitation and inhibition to characterize their differences in temporal dynamics and magnitude. By measuring the peak amplitude and the time-to-peak ($t_{\text{peak}}$), you can visualize how the specific kinetics of each synapse (e.g., fast AMPA vs. slow GABA) influence the postsynaptic response.

---

### **What to change**
*   **Synaptic Identity:** Switch between excitatory and inhibitory settings by changing the reversal potential ($E_{\text{rev}}$).
*   **Kinetics:** Adjust the decay time constant (`syn.tau`).
*   **Strength:** Modify the synaptic weight to compare responses of similar or different magnitudes.

---

### **Sanity checks**
*   **Data Summary:** Print a small table containing the following columns for both conditions:
    *   **Condition** (Excitation vs. Inhibition)
    *   **Peak Amplitude** (mV)
    *   **$t_{\text{peak}}$** (ms)

---

### **Predict → verify**
*   **Temporal Broadening:** Slower inhibitory kinetics (longer $\tau$) will significantly delay the $t_{\text{peak}}$ and broaden the total area of the IPSP compared to the sharper, faster profile of an EPSP.

---

### **Exercises**

**1. Area Equalization**
Adjust the synaptic weight of the inhibitory synapse until the total **area under the curve** (measured in $\text{mV} \cdot \text{ms}$) is approximately equal to that of a reference EPSP. 
*   Compare the resulting **peaks** and **latencies** ($t_{\text{peak}}$).
*   **Report:** Provide the values for both cases and describe the visual difference in their "shape."

**2. Kinetic Impact on Coupling**
Explain, in two or three sentences, how having an EPSP and an IPSP with equal total areas but different kinetics (one fast and sharp, one slow and broad) will differently affect **spike-timing precision** and **EPSP–spike coupling**.
*   [Your Answer Here]


In [ ]:

import numpy as np

def run_trial(e_rev,tau,w,ttl):
    syn.e=e_rev; syn.tau=tau; nc.weight[0]=w
    h.finitialize(-65*mV); h.continuerun(35*ms)
    vnp=np.array(vB); tnp=np.array(t)
    peak=vnp.max(); tpk=tnp[vnp.argmax()]
    plot_vm(t,[vB],["Neuron B"],title=f"{ttl} | peak={peak:.1f} mV @ {tpk:.2f} ms")
    return peak,tpk
p1=run_trial(0.0,2.0,0.005,'E3: EPSP')
p2=run_trial(-75.0,8.0,0.01,'E3: IPSP')
print('EPSP:',p1,'IPSP:',p2)


### **E4 — Reversal sweep**

**Idea**
Sweep synaptic reversal $E_{\text{rev}}$ to reveal where the Post-Synaptic Potential (PSP) changes sign; distinguish hyperpolarizing vs. shunting regimes.

---

### What to change
*   **Reversal Potential:** A list of $E_{\text{rev}}$ values to sweep.
*   **Synaptic Constants:** Decay time constant ($\tau$) and synaptic weight.

---

### Sanity checks
*   **Visual Sweep:** Plot the PSP traces for selected $E_{\text{rev}}$ values.
*   **Summary Data:** Produce a "peak amplitude vs. $E_{\text{rev}}$" summary plot.

---

### Predict → verify
*   **Zero Crossing:** The peak amplitude crosses the baseline near the reversal potential $E_{\text{rev}}$.
*   **Shunting Effect:** Near the resting potential, inhibitory inputs mainly reduce input resistance (apparent shunt) rather than causing large voltage deflections.

---

### Exercises

**Peak and Reversal Relation**
Show the zero crossing of the peak amplitude vs. $E_{\text{rev}}$ and relate this point to the resting membrane potential $V_m$.

**Shunting Logic**
Provide a two-sentence explanation of shunting inhibition using the synaptic current equation $I = g(V_m - E_{\text{rev}})$.
*   [Your Answer Here]


In [ ]:

import numpy as np, matplotlib.pyplot as plt

def reversal_sweep(E_values=[-85,-75,-65,-55,-45,-35,-25,-15,-5,5,10], tau=4.0, w=0.006):
    peaks=[]
    for e in E_values:
        syn.e=e; syn.tau=tau; nc.weight[0]=w
        h.finitialize(-65*mV); h.continuerun(35*ms)
        vnp=np.array(vB); tnp=np.array(t)
        peaks.append((e, vnp.max()))
        plt.figure(figsize=(5.4,2.6)); plt.plot(tnp,vnp,'k'); plt.axhline(-65,color='gray',ls='--',lw=0.8)
        plt.title(f'E4: E_syn={e} mV'); plt.xlabel('Time (ms)'); plt.ylabel('mV'); plt.grid(True,alpha=0.3); plt.tight_layout(); plt.show()
    return peaks
peaks=reversal_sweep(); print('Peak Vm by E_syn (mV):',peaks)


### **E5 — Temporal summation - two spikes, variable ISI)**

**Idea**
Two presynaptic spikes delivered at different Inter-Stimulus Intervals (ISIs) reveal how the postsynaptic response depends on the temporal overlap of synaptic conductances. As the interval between spikes decreases, the individual EPSPs sum together, increasing the total peak amplitude and area.

---

### **What to change**
*   **ISI:** Adjust the timing between the two presynaptic spikes (`ns.interval`).
*   **Synaptic Kinetics:** Modify the decay time constant (`syn.tau`).
*   **Synaptic Strength:** Adjust the weight of the connection to ensure the response remains sub-threshold.

---

### **Sanity checks**
*   **Metric Recording:** Report the resulting **peak amplitude** and **total area** (integral) for ISIs of **2 ms, 5 ms, and 20 ms**.
*   **Integration Analysis:** Note where the summation is purely **additive** (minimal overlap) versus where it begins to **saturate** or become sublinear as it approaches the reversal potential.

---

### **Predict → verify**
*   **Timing:** Shorter ISIs will lead to a significantly larger cumulative peak and area due to increased conductance overlap.
*   **Kinetics:** Increasing the synaptic time constant ($\tau$) will broaden the individual EPSPs, thereby strengthening the degree of summation at longer intervals.

---

### **Exercises**

**1. Temporal Window Mapping**
Plot the **Peak Amplitude vs. ISI**. 
*   Fit a simple exponential decay to your data points.
*   Use the decay constant from this fit to estimate the **effective temporal window** for summation in this neuron.

**2. Biophysical Constraints**
In two or three sentences, explain how this measured "summation window" relates to the interaction between the **membrane time constant** ($\tau_m$) and the **synaptic time constant** ($\tau_{syn}$).
*   [Your Answer Here]


In [ ]:

# temporal summation with NetStim

def temporal_sum(isi=5.0,e_rev=0.0,tau=2.0,w=0.005,start=5.0):
    syn.e=e_rev; syn.tau=tau
    ns=h.NetStim(); ns.number=2; ns.start=start; ns.interval=isi; ns.noise=0
    nc_ns=h.NetCon(ns,syn); nc_ns.weight[0]=w
    h.finitialize(-65*mV); h.continuerun((start+isi+15)*ms)
    plot_vm(t,[vB],["Neuron B"],title=f'E5: Temporal summation (ISI={isi} ms)')
for isi in [2.0,5.0,20.0]: temporal_sum(isi=isi)


## E6 — Spatial summation (proximal vs distal dendritic synapses) <a id='e6'></a>
### **E6 — Spatial summation (proximal vs distal)**

**Idea**  
Identical excitatory synaptic conductances are placed at two different locations: a **proximal** site (near the soma) and a **distal** site (far out on the dendrite). This demonstration reveals how dendritic morphology and passive cable properties filter and attenuate signals before they reach the soma.

---

### **What to change**
*   **Dendritic Geometry:** Adjust the total length and diameter of the dendrite section.
*   **Synaptic Placement:** Move the synapse objects to different segments along the dendrite.
*   **Weights:** Balance the synaptic weights to compare responses of varying initial strengths.

---

### **Sanity checks**
*   **Multi-Site Recording:** Plot the local dendritic $V_m$ (at the site of injection) and the somatic $V_m$ on the same graph.
*   **Signal Transformation:** Annotate the **attenuation** (loss of peak height) and the **temporal broadening** (slowing of the rise and fall) that occurs as the distal signal propagates to the soma.

---

### **Predict → verify**
*   **Proximal Advantage:** Inputs near the soma will yield larger, faster somatic EPSPs with minimal filtering.
*   **Distal Filtering:** Inputs far from the soma will appear significantly smaller and "smoother" (broadened) at the somatic recording site due to dendritic capacitance and leak.

---

### **Exercises**

**1. Morphological Influence**  
Increase the dendritic diameter (e.g., from 1.0 µm to 2.0 µm). 
*   **Measure:** Record the change in the somatic EPSP peak amplitude resulting from the **distal** synapse. 
*   **Explain:** How does a thicker dendrite alter the "space constant" ($\lambda$) and subsequent signal decay?

**2. Non-Linear Summation**  
Deliver **simultaneous** proximal and distal inputs. 
*   **Compare:** Measure the actual combined somatic peak and compare it to the **linear sum** (the peak of proximal alone + peak of distal alone). 
*   **Analyze:** Is the summation perfectly linear, or is there sublinear interference? Justify why.


In [ ]:

# add dendrite to B and compare proximal vs distal
try:
    dendB
except NameError:
    dendB=h.Section(name='dendB'); dendB.L=300; dendB.diam=1.5; dendB.Ra=100; dendB.cm=1; dendB.insert('pas'); dendB.connect(somaB(1.0))

synP=h.ExpSyn(dendB(0.2)); synP.tau=2.0; synP.e=0.0
synD=h.ExpSyn(dendB(0.8)); synD.tau=2.0; synD.e=0.0
nsP=h.NetStim(); nsP.number=1; nsP.start=6.0; nsP.noise=0
nsD=h.NetStim(); nsD.number=1; nsD.start=6.0; nsD.noise=0
ncP=h.NetCon(nsP,synP); ncP.weight[0]=0.005
ncD=h.NetCon(nsD,synD); ncD.weight[0]=0.005
v_soma=mk_rec(somaB); v_prox=h.Vector().record(dendB(0.2)._ref_v); v_dist=h.Vector().record(dendB(0.8)._ref_v)

h.finitialize(-65*mV); h.continuerun(25*ms)
plt.figure(figsize=(7,3.3))
plt.plot(t,v_prox,label='dend(0.2)'); plt.plot(t,v_dist,label='dend(0.8)'); plt.plot(t,v_soma,label='soma',color='k')
plt.xlabel('Time (ms)'); plt.ylabel('mV'); plt.title('E6: Spatial summation & attenuation'); plt.grid(True,alpha=0.3); plt.legend(); plt.tight_layout(); plt.show()


### **E7 — Feedforward inhibition (E→I)**

**Idea**
A fast EPSP followed by a delayed IPSP acts to "clamp" the voltage peak and significantly narrow the integration window for the postsynaptic neuron. This circuit motif allows for precise timing control without necessarily reducing the initial excitatory amplitude.

---

### **What to change**
*   **Synaptic Weights:** Adjust `ncE2.weight[0]` (excitatory) and `ncI2.weight[0]` (inhibitory).
*   **Timing:** Change the start time of the inhibitory stimulus relative to the excitatory one ($\Delta t = \text{nsI.start} - \text{nsE.start}$).
*   **Inhibitory Kinetics:** Modify the inhibitory time constant (`syn.tau`).

---

### **Sanity checks**
*   **Metric Sweep:** For several $\Delta t$ values, compute the **peak voltage**, **latency**, and **total area** of the postsynaptic response.
*   **Clamping Observation:** Show that as the inhibitory input approaches the excitatory input in time (smaller $\Delta t$), the voltage peak is monotonically clamped at lower levels.

---

### **Predict → verify**
*   **Window Narrowing:** A smaller $\Delta t$ (inhibition closer to excitation) results in a reduced peak and an earlier return to baseline. 
*   **Excitability:** Spike probability decreases significantly even if the EPSP onset remains unchanged, as the "window of opportunity" to reach threshold is shortened.

---

### **Exercises**

**1. Clamping Sensitivity**  
Plot the **Peak Voltage vs. $\Delta t$**. 
*   Identify and mark the specific $\Delta t$ at which the excitatory peak is reduced by **50%** compared to a purely excitatory control.

**2. Kinetic Influence on Integration**  
Keep $\Delta t$ fixed and vary the **inhibitory time constant** ($\tau$). 
*   **Explain:** Describe the differences in how "tightly" the voltage is clamped and how the decay phase of the PSP changes as inhibition becomes slower or faster.


In [ ]:

synE2=h.ExpSyn(somaB(0.5)); synE2.tau=2.0; synE2.e=0.0
synI2=h.ExpSyn(somaB(0.5)); synI2.tau=8.0; synI2.e=-75.0
nsE=h.NetStim(); nsE.number=1; nsE.start=6.0; nsE.noise=0
nsI=h.NetStim(); nsI.number=1; nsI.start=8.0; nsI.noise=0
ncE2=h.NetCon(nsE,synE2); ncE2.weight[0]=0.006
ncI2=h.NetCon(nsI,synI2); ncI2.weight[0]=0.01
v_ff=mk_rec(somaB)
h.finitialize(-65*mV); h.continuerun(25*ms)
plot_vm(t,[v_ff],["Neuron B"],title='E7: Feedforward inhibition (E→I)')


### **E8 — Analysis - latency, peak, area**

**Idea**
Provide a minimal, reusable metrics function to automatically extract key signal features such as the baseline voltage ($V_0$), peak amplitude ($V_{peak}$), time-to-peak ($t_{peak}$), and the total area under the curve (integral).

---

### **What to change**
*   **Baseline Window:** Adjust the time range used to calculate the reference $V_0$.
*   **Detection Thresholds:** Modify the criteria used to identify the start and end of a synaptic event for area calculations.

---

### **Sanity checks**
*   **Validation Run:** Execute the helper function on the most recent simulation trace.
*   **Verification:** Confirm that the output units (e.g., mV, ms) are correct and that the signs (positive for EPSPs, negative for IPSPs) match the visual data.

---

### **Exercises**

**1. Expanding the Toolkit**  
Update the metrics function to include two additional calculations:
*   **Half-width:** The duration of the signal at 50% of its peak amplitude.
*   **10–90% Rise Time:** The time it takes for the signal to transition from 10% to 90% of its peak.
*   **Compare:** Use these new metrics to quantify the specific differences between a standard EPSP and a standard IPSP from your earlier simulations.

**2. Pedagogical Justification**  
In the context of an **E→I (Excitation-Inhibition)** lab, decide which two metrics are the most critical to grade. 
*   **Justify:** Explain why these specific metrics are the best indicators of a student's understanding of synaptic integration and feedforward circuits.


In [ ]:

import numpy as np

def psp_metrics(tvec,vtrace,baseline=(0,5)):
    tnp=np.array(tvec); vnp=np.array(vtrace)
    b=(tnp>=baseline[0])&(tnp<=baseline[1]); v0=vnp[b].mean() if b.any() else vnp[0]
    peak=vnp.max(); tpk=tnp[vnp.argmax()]
    area=np.trapz(vnp-v0,tnp)
    return {'V0':v0,'Vpeak':peak,'tpeak_ms':tpk,'Area_mV_ms':area}
print('E8 metrics (last trace on vB):', psp_metrics(t,vB))


### **E9 — Playground- tweak synaptic parameters and rerun**

**Idea**
Free exploration: systematically vary synaptic parameters to practice **predict → verify** reasoning. This section is designed to test your intuition about how electrochemical gradients and channel kinetics shape the post-synaptic response.

---

### **What to change**
*   **Synaptic Identity:** Reversal potential ($E_{\text{rev}}$).
*   **Temporal Profile:** Decay time constant ($\tau$).
*   **Strength:** Synaptic weight.
*   **Timing:** Axonal or stimulus delay.
*   **Documentation:** Record the specific parameter tuple used for every run.

---

### **Sanity checks**
For every experimental configuration, consistently record the following metrics:
*   **Baseline voltage** ($V_0$)
*   **Peak amplitude** ($V_{\text{peak}}$)
*   **Latency to peak** ($t_{\text{peak}}$)
*   **Total area** (integral)

---

### **Exercises**

**1. Predict → Verify Log**
Perform three distinct "custom" runs. For each, write down your prediction of the change (e.g., "Increasing $\tau$ while decreasing weight will result in a wider but shorter PSP") before executing the code.
*   **Run 1:** [Parameters] -> [Prediction] -> [Result]
*   **Run 2:** [Parameters] -> [Prediction] -> [Result]
*   **Run 3:** [Parameters] -> [Prediction] -> [Result]

**2. Synthesis Question**
Based on your playground explorations, which single parameter change had the most non-linear effect on the **total area** of the synaptic response? Justify your answer.
*   [Your Answer Here]


In [ ]:

syn.e=0.0; syn.tau=2.0; nc.weight[0]=0.008; nc.delay=0.5
h.finitialize(-65*mV); h.continuerun(30*ms)
plot_vm(t,[vA,vB],["Neuron A","Neuron B"],title='E9: Playground run')


## Reflection <a id='reflection'></a>
- How do E_syn and τ jointly control PSP amplitude and timing?
- When does inhibitory input look depolarizing yet stay functionally inhibitory?
- How do ISI and electrotonic distance interact in postsynaptic integration?

## Practice / Discussion Questions — Set B — Biology → Model Mapping (20)

1) Define a **model abstraction** for a neuron that preserves interpretability: which parameters must be in **biophysical units**, and why?
2) Explain how **blocks** (RC membrane, synaptic conductance, spike generator) can be composed into a testable single‑cell model.
3) *Justify*: What’s gained and lost when moving from detailed HH‑type to reduced spiking models?
4) How do you **document assumptions** so a reader can reproduce and critique your model? (List 3–4 items.)
5) What’s a **sanity check** for a synapse implemented with a fixed reversal potential? (Describe the test and expected result.)
6) Outline a minimal **reproducible workflow** (notebook → plots → metrics) for a membrane step test.
7) Why is **parameter transparency** essential for educational modeling? Give one consequence of opaque scaling.
8) *Design*: Write 3 summary metrics you would log for “EPSP → spike coupling” experiments.
9) Choose one **modeling decision** (e.g., omit Ca²⁺) and argue when it’s acceptable for instruction.
10) What is the **role of NEURON** in enabling reproducibility for single cells and small networks?
11) *Compare*: When do you favor **current clamp** vs **conductance injection** to test a component?
12) Define a **minimal dendrite** you would include to capture location effects without overcomplicating the model.
13) What’s the educational **trade‑off** of adding noise sources to a beginner model?
14) How would you **validate** your inhibitory synapse timing motif before using it in a network?
15) *Scenario*: Your EPSPs look too broad. List 3 plausible causes and the diagnostic you’d run.
16) Why do we advocate **fixed random seeds** in notebooks for teaching?
17) Provide a **short rubric** for students to self‑assess whether their model description is reproducible by others.
18) Suggest two **figures** (plots/tables) that best communicate a membrane and synapse validation.
19) Give an example of a **missing mechanism** signaled by a consistent mismatch between model and data.
20) *Reflect*: What will you carry from Set B into Set C–E when you begin analyzing spatial effects?



---

## Practice / Discussion Questions — Set E — Synapses & Transmission

1) Explain how **fixed reversal potentials** make synaptic efficacy **state‑dependent**.
2) Define and compare **AMPA‑like** vs **GABA_A** synapses in terms of E_rev and kinetics.
3) *Predict*: How will holding at −50 vs −70 mV change the apparent sign/magnitude of a GABA_A PSP?
4) Describe the **E→I feedforward motif** and its effect on graded→spike conversion.
5) Define a **timing window** (E leads I by Δt) and predict effects on peak and latency.
6) Give one **temporal** and one **spatial** summation example; state what changes.
7) *Reasoning*: Why do distal excitatory inputs often require cooperation to impact the soma?
8) What does **shunting inhibition** mean and how is it revealed experimentally?
9) Name two **metrics** to evaluate strength/timing of synaptic integration.
10) Sketch a protocol to verify that E→I reduces spike probability without changing E magnitude.
11) How does membrane **state** (input resistance, V_m) gate synaptic impact?
12) Distinguish **incidence of local events** vs **coupling** to soma; give an example.
13) *Critique*: Risk of using only somatic measures for synaptic efficacy.
14) Why does inhibitory **location** (perisomatic vs dendritic) matter for timing and gain?
15) Provide a case where inhibitory timing **facilitates** information transmission; explain.
16) *Compute/Explain*: How does driving force change as V_m approaches E_rev for an inhibitory synapse?
17) Quick **two‑condition test** to separate shunting from hyperpolarizing inhibition.
18) If two EPSPs overlap, when expect **superlinear** vs **sublinear** summation? Why?
19) Connect Set E outcomes to **network persistence** (why timing matters).
20) State the minimal integration insights from E you’ll carry to Set F.
